# Relegation struggles — data collection (1)

Goal: list the **two teams relegated** each season using the rules below, as a baseline for later “relegation battle” analysis.

## Where the numbers come from

- **12-team seasons** (2006/07–2008/09): **two** clubs relegate → take **11th and 12th** from `regular_final_table_*` (end of the regular double round; no split league in this project’s data).

- **16- and 14-team seasons** (split league): **two** clubs relegate after the **relegation (bottom) playoff**. The files `data/interim/scraped_standings/playoff_final_table_*.csv` in this repo are **championship (top) group** tables only — they do **not** contain the bottom group’s final standings. Here we build that **bottom-group final table** the same way the league does: **carry** points and goals from `regular_final_table_*` for every team that appears in the relegation playoff, then **add** match results from `data/raw/playoffs_relegation_*_ligat_haal_wikipedia.csv`. The **two worst** teams on that combined table (points, then goal difference, then goals scored) are taken as relegated.

Round-by-round “who was in the fight from round X?” can use `data/processed/position_tables_by_round_tm/` in a follow-up notebook.

**Leader → last safe** (points gap to the club just above the drop zone) is built in a **separate table** below — one row per season, saved as `leader_to_last_safe_gap_by_season.csv`.

In [15]:
from pathlib import Path
from typing import Optional
import re
import pandas as pd


def find_repo_root(start: Optional[Path] = None) -> Path:
    p = start or Path.cwd()
    for _ in range(6):
        if (p / "data").exists() or (p / ".git").exists() or (p / "notebooks").exists():
            if not (p / "data").exists() and (p.parent / "data").exists():
                p = p.parent
            return p
        p = p.parent
    return Path.cwd()


ROOT = find_repo_root()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
INTERIM_DIR = ROOT / "data" / "interim"
PROCESSED_DIR = ROOT / "data" / "processed"
DATA_DIR = ROOT / "data" / "raw"
REGULAR_FINAL_DIR = INTERIM_DIR / "scraped_standings" / "regular_final_tables"
OUT_DIR = INTERIM_DIR / "relegation_struggles"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT: {ROOT}")
print(f"Regular final tables: {REGULAR_FINAL_DIR}")
print(f"Output dir: {OUT_DIR}")

ROOT: c:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks
Regular final tables: c:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\interim\scraped_standings\regular_final_tables
Output dir: c:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\interim\relegation_struggles


In [16]:
def season_from_filename(name: str) -> Optional[str]:
    m = re.match(r"regular_final_table_(\d{4})_(\d{2})\.csv$", name, re.I)
    if not m:
        return None
    y1, y2 = int(m.group(1)), m.group(2)
    return f"{y1}/{y2}"


WIKI_TO_TM_MAP: dict[str, str] = {
    "BNS": "Bnei Sakhnin",
    "Beitar Jerusalem": "B. Jerusalem",
    "Bnei Sakhnin": "Bnei Sakhnin",
    "Bnei Yehuda": "Bnei Yehuda",
    "F.C. Ashdod": "FC Ashdod",
    "Hapoel Acre": "Hapoel Acre",
    "Hapoel Ashkelon": "H. Ashkelon",
    "Hapoel Be'er Sheva": "H. Beer Sheva",
    "Hapoel Hadera": "Hapoel Hadera",
    "Hapoel Haifa": "Hapoel Haifa",
    "Hapoel Jerusalem": "H. Jerusalem",
    "Hapoel Kfar Saba": "H. Kfar Saba",
    "Hapoel Nof HaGalil": "H. Nof HaGalil",
    "Hapoel Petah Tikva": "H. Petah Tikva",
    "Hapoel Ra'anana": "Hapoel Raanana",
    "Hapoel Ramat Gan": "H. Ramat Gan",
    "Hapoel Ramat HaSharon": "Ramat haSharon",
    "Hapoel Tel Aviv": "Hapoel Tel Aviv",
    "Ironi Kiryat Shmona": "Kiryat Shmona",
    "Ironi Ramat HaSharon": "Ramat haSharon",
    "Ironi Tiberias": "Ironi Tiberias",
    "Maccabi Ahi Nazareth": "M. Ahi Nazareth",
    "Maccabi Bnei Reineh": "M. Bnei Reineh",
    "Maccabi Haifa": "Maccabi Haifa",
    "Maccabi Netanya": "Maccabi Netanya",
    "Maccabi Petah Tikva": "M. Petah Tikva",
    "Ness Ziona": "Ness Ziona",
    "Rishon LeZion": "H Rishon leZion",
    "Sektzia Ness Ziona": "Ness Ziona",
}


def parse_goals(cell: object) -> tuple[int, int]:
    s = str(cell)
    if ":" not in s:
        return 0, 0
    a, b = s.split(":", 1)
    return int(a.strip()), int(b.strip())


def wiki_relegation_csv(season: str) -> Path:
    safe = season.replace("/", "_")
    return DATA_DIR / f"playoffs_relegation_{safe}_ligat_haal_wikipedia.csv"


def resolve_wiki_team_to_tm(wiki_name: str, reg_teams: set[str]) -> str:
    cand = WIKI_TO_TM_MAP.get(wiki_name, wiki_name)
    if cand in reg_teams:
        return cand
    if wiki_name in reg_teams:
        return wiki_name
    lower = {t.lower(): t for t in reg_teams}
    if cand.lower() in lower:
        return lower[cand.lower()]
    if wiki_name.lower() in lower:
        return lower[wiki_name.lower()]
    raise KeyError(f"Wikipedia name {wiki_name!r} → {cand!r} not found in regular table teams")


def relegation_pool_final_table(season: str, reg: pd.DataFrame, wiki: pd.DataFrame) -> pd.DataFrame:
    """Relegation mini-league final: carried regular points/goals + relegation playoff matches. Best team first."""
    reg_teams = set(reg["team"].astype(str))
    pool_wiki = set(wiki["home_team"]) | set(wiki["away_team"])
    pool_tm = {w: resolve_wiki_team_to_tm(w, reg_teams) for w in pool_wiki}

    stats: dict[str, dict[str, int]] = {}
    for tm in set(pool_tm.values()):
        sub = reg[reg["team"] == tm]
        if len(sub) != 1:
            raise ValueError(f"{season}: expected one regular row for {tm!r}, got {len(sub)}")
        row = sub.iloc[0]
        gf, ga = parse_goals(row["goals"])
        stats[tm] = {"points": int(row["points"]), "gf": gf, "ga": ga}

    for _, r in wiki.iterrows():
        ht = pool_tm[r["home_team"]]
        at = pool_tm[r["away_team"]]
        stats[ht]["gf"] += int(r["home_goals"])
        stats[ht]["ga"] += int(r["away_goals"])
        stats[ht]["points"] += int(r["home_points"])
        stats[at]["gf"] += int(r["away_goals"])
        stats[at]["ga"] += int(r["home_goals"])
        stats[at]["points"] += int(r["away_points"])

    rows = []
    for tm, s in stats.items():
        gd = s["gf"] - s["ga"]
        rows.append({"team": tm, "points": s["points"], "goal_diff": gd, "gf": s["gf"]})
    tdf = pd.DataFrame(rows)
    return tdf.sort_values(["points", "goal_diff", "gf"], ascending=[False, False, False]).reset_index(drop=True)


def relegated_after_relegation_playoff(season: str, reg: pd.DataFrame, wiki: pd.DataFrame) -> pd.DataFrame:
    """Two relegated teams (worst first)."""
    pool = relegation_pool_final_table(season, reg, wiki)
    return pool.tail(2).sort_values(["points", "goal_diff", "gf"], ascending=[True, True, True]).reset_index(drop=True)


def first_place_points_playoff_final(season: str) -> int:
    """Points for championship rank 1 from scraped playoff final table."""
    safe = season.replace("/", "_")
    path = PLAYOFF_FINAL_DIR / f"playoff_final_table_{safe}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing playoff final table for {season}: {path.name}")
    df = pd.read_csv(path)
    if "season" in df.columns:
        df = df[df["season"].astype(str) == season].copy()
    top = df.loc[df["rank"] == 1, "points"]
    if top.empty:
        raise ValueError(f"No rank 1 row in {path.name}")
    return int(top.iloc[0])

In [17]:
rows: list[dict] = []
files = sorted(REGULAR_FINAL_DIR.glob("regular_final_table_*.csv"))
if not files:
    raise FileNotFoundError(f"No regular_final_table_*.csv under {REGULAR_FINAL_DIR}")

for path in files:
    season = season_from_filename(path.name)
    if not season:
        continue
    reg = pd.read_csv(path)
    n_teams = len(reg)

    if n_teams == 12:
        r1 = reg.loc[reg["rank"] == 1].iloc[0]
        r_safe = reg.loc[reg["rank"] == n_teams - 2].iloc[0]
        p_first = int(r1["points"])
        p_safe = int(r_safe["points"])
        t_safe = r_safe["team"]
        gap_leader_safe = p_first - p_safe
        bottom = reg.sort_values("rank", ascending=False).head(2)
        for i, (_, r) in enumerate(bottom.iterrows(), start=1):
            rows.append(
                {
                    "season": season,
                    "league_size": n_teams,
                    "method": "regular_table_bottom_2",
                    "regular_rank": int(r["rank"]),
                    "team": r["team"],
                    "points_after": int(r["points"]) if pd.notna(r["points"]) else None,
                    "goal_diff_after": int(r["goal_diff"]) if pd.notna(r["goal_diff"]) else None,
                    "relegation_pool_rank_worst_first": i,
                    "regular_final_file": path.name,
                    "relegation_wiki_file": None,
                    "points_first_place": p_first,
                    "team_last_safe_above_relegation": t_safe,
                    "points_last_safe_above_relegation": p_safe,
                    "gap_points_leader_to_last_safe": gap_leader_safe,
                }
            )
        continue

    if n_teams not in (14, 16):
        raise ValueError(f"{season}: unexpected league_size {n_teams}")

    wiki_path = wiki_relegation_csv(season)
    if not wiki_path.exists():
        raise FileNotFoundError(f"Missing relegation playoff CSV for {season}: {wiki_path.name}")

    wiki = pd.read_csv(wiki_path)
    wiki = wiki[wiki["playoff_type"] == "relegation"].copy()
    if wiki.empty:
        raise ValueError(f"{season}: no relegation rows in {wiki_path.name}")

    worst2 = relegated_after_relegation_playoff(season, reg, wiki)
    for i, (_, r) in enumerate(worst2.iterrows(), start=1):
        rows.append(
            {
                "season": season,
                "league_size": n_teams,
                "method": "relegation_playoff_carried_plus_wiki",
                "regular_rank": None,
                "team": r["team"],
                "points_after": int(r["points"]),
                "goal_diff_after": int(r["goal_diff"]),
                "relegation_pool_rank_worst_first": i,
                "regular_final_file": path.name,
                "relegation_wiki_file": wiki_path.name,
            }
        )

relegated_df = pd.DataFrame(rows).sort_values(["season", "relegation_pool_rank_worst_first"]).reset_index(drop=True)
relegated_df

,season,league_size,method,regular_rank,team,points_after,goal_diff_after,relegation_pool_rank_worst_first,regular_final_file,relegation_wiki_file,points_first_place,team_last_safe_above_relegation,points_last_safe_above_relegation,gap_points_leader_to_last_safe
0,2006/07,12,regular_table_bottom_2,12.0,H. Petah Tikva,16,-32,1,regular_final_table_2006_07.csv,None,67.0,Maccabi Herzlya,34.0,33.0
1,2006/07,12,regular_table_bottom_2,11.0,Hakoah Amidar,28,-22,2,regular_final_table_2006_07.csv,None,67.0,Maccabi Herzlya,34.0,33.0
2,2007/08,12,regular_table_bottom_2,12.0,Maccabi Herzlya,30,-19,1,regular_final_table_2007_08.csv,None,67.0,M. Petah Tikva,37.0,30.0
3,2007/08,12,regular_table_bottom_2,11.0,H. Kfar Saba,37,-17,2,regular_final_table_2007_08.csv,None,67.0,M. Petah Tikva,37.0,30.0
4,2008/09,12,regular_table_bottom_2,12.0,Kiryat Shmona,27,-20,1,regular_final_table_2008_09.csv,None,67.0,H. Petah Tikva,31.0,36.0
5,2008/09,12,regular_table_bottom_2,11.0,Hakoah Amidar,29,-20,2,regular_final_table_2008_09.csv,None,67.0,H. Petah Tikva,31.0,36.0
6,2009/10,16,relegation_playoff_carried_plus_wiki,NaN,M. Ahi Nazareth,28,-48,1,regular_final_table_2009_10.csv,playoffs_relegation_2009_10_ligat_haal_wikiped...,NaN,NaN,NaN,NaN
7,2009/10,16,relegation_playoff_carried_plus_wiki,NaN,Hapoel Raanana,28,-25,2,regular_final_table_2009_10.csv,playoffs_relegation_2009_10_ligat_haal_wikiped...,NaN,NaN,NaN,NaN
8,2010/11,16,relegation_playoff_carried_plus_wiki,NaN,H. Ramat Gan,18,-41,1,regular_final_table_2010_11.csv,playoffs_relegation_2010_11_ligat_haal_wikiped...,NaN,NaN,NaN,NaN
9,2010/11,16,relegation_playoff_carried_plus_wiki,NaN,H. Ashkelon,32,-33,2,regular_final_table_2010_11.csv,playoffs_relegation_2010_11_ligat_haal_wikiped...,NaN,NaN,NaN,NaN


In [19]:
out_path = OUT_DIR / "relegated_teams_after_playoffs.csv"
relegated_df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Saved: {out_path} ({len(relegated_df)} rows)")

Saved: c:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\interim\relegation_struggles\relegated_teams_after_playoffs.csv (38 rows)
